In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

from IPython.display import clear_output
import time

from typing import Literal, Tuple

# NO AMBIENTE:
#   1 = PAREDE
#   2 = SUJEIRA
#   0 = NEUTRO

In [ ]:
def exibir(ambiente, pos: Tuple[int, int]):
  x, y = pos
  plt.imshow(ambiente, cmap='gray')
  plt.plot(y, x, 'ro')
  plt.show()
  plt.pause(0.5)

def sujar(ambiente, qdte_sujeira: int):
  for i in range(1, qdte_sujeira):
    x = random.randint(1, 4)
    y = random.randint(1, 4)
    ambiente[x][y] = 2

def parede(x: int, y: int) -> bool:
  return ambiente[x][y] == 1

ACAO = Literal['acima', 'abaixo', 'esquerda', 'direita', 'aspirar']

def mapeamento(pos: Tuple[int, int], acao: str) -> Tuple[int, int]:
  x, y = pos

  movimentos = {
    'acima': (-1, 0),
    'abaixo': (1, 0),
    'esquerda': (0, -1),
    'direita': (0, 1)
  }

  oposto = {
    'acima': 'abaixo',
    'abaixo': 'acima',
    'esquerda': 'direita',
    'direita': 'esquerda'
  }

  def movimento_valido(nx, ny):
    return (
      0 <= nx < len(ambiente) and
      0 <= ny < len(ambiente[0]) and
      ambiente[nx][ny] != 1
    )

  if acao == 'aspirar':
    return (x, y)

  tentativas = [acao, oposto[acao]]

  for m in movimentos:
    if m not in tentativas:
      tentativas.append(m)

  for tentativa in tentativas:
    dx, dy = movimentos[tentativa]
    nx, ny = x + dx, y + dy

    if movimento_valido(nx, ny):
      return (nx, ny)

  return pos

def gerarCaminho():
  caminho = []
  for i in range(1, 5):
    if i % 2 == 1:
      for j in range(1, 5):
        caminho.append((i, j))
    else:
      for j in range(4, 0, -1):
        caminho.append((i, j))
  return caminho

In [ ]:
PERCEPCAO_TIPOS = Literal[0, 2]

caminho = gerarCaminho()
indice_caminho = 0

def agenteReativoSimples(pos: Tuple[int, int], percepcao: PERCEPCAO_TIPOS) -> ACAO:
  global indice_caminho

  if percepcao == 2:
    return 'aspirar'

  destino = caminho[indice_caminho]

  if pos == destino:
    indice_caminho = (indice_caminho + 1) % len(caminho)
    destino = caminho[indice_caminho]

  x, y = pos
  dx, dy = destino

  if dx < x: acao = 'acima'
  elif dx > x: acao = 'abaixo'
  elif dy < y: acao = 'esquerda'
  elif dy > y: acao = 'direita'
  else: acao = 'aspirar'

  return acao

ambiente = [
    [1,1,1,1,1,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,1,1,1,1,1]
]

sujar(ambiente, 6)

agente_pos_x, agente_pos_y = 1, 1

while True:
    exibir(ambiente, (agente_pos_x, agente_pos_y))

    percepcao = ambiente[agente_pos_x][agente_pos_y]
    acao = agenteReativoSimples((agente_pos_x, agente_pos_y), percepcao)

    if acao == 'aspirar':
      ambiente[agente_pos_x][agente_pos_y] = 0
    else:
      agente_pos_x, agente_pos_y = mapeamento((agente_pos_x, agente_pos_y), acao)

In [ ]:
def checkObj(sala):
  for linha in sala:
    for celula in linha:
      if celula == 2:
        return 1
  return 0

caminho = gerarCaminho()
indice_caminho = 0
pontos = 0

def agenteObjetivo(pos, percepcao, objObtido):
  global indice_caminho, pontos

  if percepcao == 2:
    pontos += 1
    print(f"Estado da percepcao: {percepcao} Acao escolhida: aspirar")
    return 'aspirar'

  if objObtido == 0:
    print(f"Ponto: -> {pontos}")
    return 'NoOp'

  destino = caminho[indice_caminho]

  if pos == destino:
    indice_caminho = (indice_caminho + 1) % len(caminho)
    destino = caminho[indice_caminho]

  x, y = pos
  dx, dy = destino

  if dx < x:
    acao = 'acima'
  elif dx > x:
    acao = 'abaixo'
  elif dy < y:
    acao = 'esquerda'
  elif dy > y:
    acao = 'direita'
  else:
    acao = 'NoOp'

  pontos += 1
  print(f"Estado da percepcao: {percepcao} Acao escolhida: {acao}")

  return acao

ambiente = [
    [1,1,1,1,1,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,1,1,1,1,1]
]

agente_pos_x, agente_pos_y = 1, 1

sujar(ambiente, 6)

while True:

  exibir(ambiente, (agente_pos_x, agente_pos_y))
  percepcao = ambiente[agente_pos_x][agente_pos_y]
  objObtido = checkObj(ambiente)

  acao = agenteObjetivo((agente_pos_x, agente_pos_y), percepcao, objObtido)

  if acao == 'NoOp':
    break

  if acao == 'aspirar':
    ambiente[agente_pos_x][agente_pos_y] = 0
  else:
    x, y = mapeamento((agente_pos_x, agente_pos_y), acao)
    agente_pos_x, agente_pos_y = x, y


## Responda: A) A sua solução é extensível para um mundo 3 x 3? E para um mundo 6 x 6? Explique sua resposta.

Para um mundo 3×3:
Não é diretamente extensível. Isso porque a função gerarCaminho() foi construída fixamente para percorrer posições de (1 até 4) , ou seja, ela assume um ambiente interno de tamanho 4×4 (desconsiderando as paredes).
Em um mundo 3×3, o caminho gerado incluiria posições que não existem, causando erro ou comportamento incorreto.

Para um mundo 6×6:
Também não é diretamente extensível pelo mesmo motivo. O caminho continua limitado ao intervalo fixo (1 a 4), então o agente não percorre todo o ambiente, deixando partes sem visita.

Conclusão:
A solução não é escalável para outros tamanhos de mundo, pois o percurso do agente é fixo. Para torná-la extensível, seria necessário gerar o caminho dinamicamente.

In [ ]:
# EXEMPLO DO ERRO

ambiente = [
    [1,1,1,1,1],
    [1,0,0,0,1],
    [1,0,0,0,1],
    [1,0,0,0,1],
    [1,1,1,1,1]
]

sujar(ambiente, 6)

agente_pos_x, agente_pos_y = 1, 1

while True:
    exibir(ambiente, (agente_pos_x, agente_pos_y))

    percepcao = ambiente[agente_pos_x][agente_pos_y]
    acao = agenteReativoSimples((agente_pos_x, agente_pos_y), percepcao)

    if acao == 'aspirar':
      ambiente[agente_pos_x][agente_pos_y] = 0
    else:
      agente_pos_x, agente_pos_y = mapeamento((agente_pos_x, agente_pos_y), acao)

In [ ]:
# CORREÇÕES NO CÓDIGO PARA MUNDOS NxN (3x3, 6x6, etc)

def sujar(ambiente, qdte_sujeira: int):
  for i in range(1, qdte_sujeira):
    x = random.randint(1, 3)
    y = random.randint(1, 3)
    ambiente[x][y] = 2

def gerarCaminho(ambiente):
    caminho = []
    linhas = len(ambiente)
    colunas = len(ambiente[0])

    for i in range(1, linhas - 1):
        if i % 2 == 1:
            for j in range(1, colunas - 1):
                caminho.append((i, j))
        else:
            for j in range(colunas - 2, 0, -1):
                caminho.append((i, j))

    return caminho

caminho = gerarCaminho(ambiente)
indice_caminho = 0

def agenteReativoSimples(pos, percepcao):
    global indice_caminho, caminho

    if percepcao == 2:
        return 'aspirar'

    destino = caminho[indice_caminho]

    if pos == destino:
        indice_caminho = (indice_caminho + 1) % len(caminho)
        destino = caminho[indice_caminho]

    x, y = pos
    dx, dy = destino

    if dx < x:
        return 'acima'
    elif dx > x:
        return 'abaixo'
    elif dy < y:
        return 'esquerda'
    elif dy > y:
        return 'direita'
    else:
        return 'aspirar'

ambiente = [
    [1,1,1,1,1],
    [1,0,0,0,1],
    [1,0,0,0,1],
    [1,0,0,0,1],
    [1,1,1,1,1]
]

sujar(ambiente, 6)

agente_pos_x, agente_pos_y = 1, 1

while True:
    exibir(ambiente, (agente_pos_x, agente_pos_y))

    percepcao = ambiente[agente_pos_x][agente_pos_y]
    acao = agenteReativoSimples((agente_pos_x, agente_pos_y), percepcao)

    if acao == 'aspirar':
      ambiente[agente_pos_x][agente_pos_y] = 0
    else:
      agente_pos_x, agente_pos_y = mapeamento((agente_pos_x, agente_pos_y), acao)

## Responda: B) É possível ter todo o espaço limpo efetivamente? Justifique sua resposta.

Sim, é possível, pois o agente continua agindo enquanto houver sujeira (checkObj). Ele só para quando tudo estiver limpo. Porém, isso só funciona bem se o caminho percorrer todo o ambiente; caso contrário, pode deixar sujeira sem limpar.